# Demo 4 · The red pull request

**Slides:** "Test results, not exit codes" (slide 16), demo step 04.

A golden-result test trains a small seeded model and compares the accuracy with a committed value.
Here we make a change that looks harmless, a "tidier" preprocessing step, and watch the test catch it.
On GitHub the same test runs in `.github/workflows/golden.yml` on every pull request.

Runtime: CPU. Run the setup cells below first: about 2 minutes plus one session restart.

## 0 · Setup (run once per session)

Each Colab notebook has its own session, so run these cells even if you ran notebook 01. Restart the session when the install cell says so, then continue from the cell after it.

In [ ]:
REPO_URL = "https://github.com/keila-moral/reproducible-ml-demo"
import os
if not os.path.exists("/content/repo"):
    !git clone -q {REPO_URL} /content/repo
else:
    !git -C /content/repo pull -q   # already cloned in this session: get the latest version
%cd /content/repo
!git log --oneline -1

In [ ]:
# Install exactly what uv.lock records, verified by sha256 hash.
!pip install -q uv
!uv export --frozen --no-emit-project --format requirements-txt -o requirements.lock.txt
!uv pip install --system -q --require-hashes -r requirements.lock.txt
!uv pip install --system -q --no-deps .
# Colab's preinstalled bottleneck is too old for the locked pandas and only causes warnings.
!uv pip uninstall --system -q bottleneck
print("Done. Now restart the session: Runtime > Restart session, then continue below.")

In [ ]:
%cd /content/repo
import os, sys
os.environ["KERAS_BACKEND"] = "jax"
# Make the demo code importable here and in commands such as !dvc repro and !pytest.
os.environ["PYTHONPATH"] = "/content/repo/src"
sys.path.insert(0, "/content/repo/src")

## 1 · The golden value and the test

In [ ]:
!cat tests/golden/golden.json
!cat tests/golden/test_result.py

## 2 · On the main branch: green

First the plain-language check, then the same comparison as CI runs it with pytest.

In [ ]:
!dvc repro -q
!python -m repro_demo.check
!python -m pytest tests/golden -q -p no:warnings

## 3 · A harmless-looking refactor

Replace the fixed `/ 16` scaling with per-batch standardisation. No error, no warning, and a reviewer might approve it.

In [ ]:
from pathlib import Path

TRAIN = Path("src/repro_demo/train.py")
ORIGINAL = 'return features.astype("float32") / 16.0'
REFACTOR = 'features = features.astype("float32")\n    return (features - features.mean()) / features.std()'

def apply_refactor():
    TRAIN.write_text(TRAIN.read_text().replace(ORIGINAL, REFACTOR))
    print("Refactor applied to src/repro_demo/train.py:")

apply_refactor()
!git diff src/

## 4 · The test catches it: red

In [ ]:
!python -m repro_demo.check
!python -m pytest tests/golden -q --tb=line -p no:warnings

Look at the numbers: the new accuracy is **higher** than the golden value, and the test still fails.
That is the point. The test does not judge whether the change is good; it says **the result moved,
and nobody said it would**. A red golden test asks one question: *was this change intended?*
There are two honest answers, and both end green.

## 5a · Not intended: undo the refactor

The refactor was meant to be cosmetic. Undo it and check the result again.

In [ ]:
!git checkout -- src/
!git diff --quiet src/ && echo "Refactor undone: src/ is identical to the last commit."
print()
!python -m repro_demo.check

## 5b · Intended: accept the new result, in the open

Suppose standardisation is a real improvement. Then the golden value is updated **in the same pull request**,
so the reviewer sees the code change and the result change side by side.

In [ ]:
apply_refactor()
!git diff --stat src/
print()
print("Updating the golden value to this code's result:")
!python -m repro_demo.make_golden
print("What the reviewer sees in the pull request:")
!git diff tests/golden/golden.json
print()
!python -m repro_demo.check

The diff of `golden.json` is the important part: the pull request now says, in plain numbers,
"this change moves accuracy from 0.958 to 0.972". A reviewer can accept that on purpose.

## 6 · Reset for the next run

In [ ]:
!git checkout -- src/ tests/
!git diff --quiet && echo "Reset: all code and the golden value are back to the last commit."

## What to say

The code ran in every case. Only a check on the **result** noticed that the number moved, and it forced a
decision: undo, or accept the new number in the open. In CI this runs whenever `uv.lock`, the source,
or the data definition changes, including dependency updates opened by Dependabot.